In [62]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV, GroupShuffleSplit
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer

In [17]:
df = pd.read_csv('mes_donnees_normalisées.csv', sep=';')

print(df.head())
df.columns = df.columns.str.strip()
#Modification de la colonne Lt pour la transformer en variable numérique pour le random Forest
# 1. Nettoyage des espaces invisibles

df['Lt'] = df['Lt'].astype(str).str.strip()
print(df)
# 2. Conversion en valeurs numériques (G -> 1, R -> 0)
df['Lt'] = df['Lt'].map({'G': 1, 'R': 0})

# 3. On ne garde que les pièces de type R
df = df[df['Lt'] == 0]



      essai  P6_reel        E1    V        E2   P1_reel  Lq1_reel  Lq2_reel  \
0  392291.0      3.2  0.508681  0.0  0.762007  0.531993  0.140432  0.086909   
1  392291.0      3.2  0.480526  0.0  0.599283  0.531993  0.063272  0.053905   
2  392291.0      3.2  0.528390  0.0  0.675269  0.531993  0.078704  0.042904   
3  392291.0      3.2  0.481933  0.0  0.620072  0.531993  0.109568  0.053905   
4  392291.0      3.2  0.535429  0.0  0.750538  0.531993  0.140432  0.086909   

    Lq_reel  Lq_target_x  ...    C   P2  Lt  Lq_target_y   P3        P4   P5  \
0  0.139073     0.105263  ...  0.0  0.0 NaN          0.0  0.5  0.646018  0.0   
1  0.056291     0.105263  ...  0.0  0.0 NaN          0.0  0.5  0.646018  0.0   
2  0.072848     0.105263  ...  0.0  0.0 NaN          0.0  0.5  0.646018  0.0   
3  0.089404     0.105263  ...  0.0  0.0 NaN          0.0  0.5  0.646018  0.0   
4  0.139073     0.105263  ...  0.0  0.0 NaN          0.0  0.5  0.646018  0.0   

   P6_target     S_moy   sigma_S  
0        

In [57]:
# 1. Création de df_ut (avec .copy() !)
# On a RETIRÉ 'Lt' des features car elle vaut 'G' partout maintenant, elle ne sert plus à rien.
df_ut = df[['P1_reel','P2','Lq_reel','P4','P5','E2','P6_reel','E1']].copy()


# 2. Remplacement des NaN (sur df_ut uniquement !)
p1_v0_par_essai = df[df['V'] == 0].groupby('essai')['P1_reel'].first()
df_ut['P1_reel'] = df['essai'].map(p1_v0_par_essai).fillna(df['P1_reel'])

lq_valide_par_essai = df.dropna(subset=['Lq_reel']).groupby('essai')['Lq_reel'].first()
df_ut['Lq_reel'] = df['essai'].map(lq_valide_par_essai).fillna(df['Lq_reel'])

# 3. Nettoyage final
df_ut = df_ut.dropna()
print(f"Nombre de lignes après dropna(): {df_ut.shape[0]}")

df_ut['essai'] = df['essai']  # Ajout de la colonne 'essai' pour le GroupShuffleSplit

X1 = df_ut.drop(columns=['E1'])
y1 = df_ut[['essai', 'E1']]

# 4. Split par groupe
groupes = X1['essai']
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(X1, y1, groups=groupes))

X_train, X_test = X1.iloc[train_idx].copy(), X1.iloc[test_idx].copy()
y_train, y_test = y1.iloc[train_idx].copy(), y1.iloc[test_idx].copy()
groupes_train = groupes.iloc[train_idx]

# 5. On retire l'essai pour ne pas tricher
X_train = X_train.drop(columns=['essai'])
X_test = X_test.drop(columns=['essai'])
y_test = y_test['E1']
y_train = y_train['E1']


Nombre de lignes après dropna(): 0


ValueError: Found array with 0 sample(s) (shape=(0,)) while a minimum of 1 is required.

In [13]:
#Préparation des données 

# Séparation des features et de la target
df_ut = df[['P1_reel','P2','Lt','Lq_reel','P4','P5','V','E2','P6_reel','E1']]

#On remplace les valeurs manquantes de P1_reel et Lq_reel par la première valeur non nulle de chaque essai selon les conseils de l'encadrante
#Cela permet de conserver d'avantage de lignes avant le dropna()
p1_v0_par_essai = df[df['V'] == 0].groupby('essai')['P1_reel'].first()
df_ut['P1_reel'] = df['essai'].map(p1_v0_par_essai).fillna(df['P1_reel'])

lq_valide_par_essai = df.dropna(subset=['Lq_reel']).groupby('essai')['Lq_reel'].first()
df_ut['Lq_reel'] = df['essai'].map(lq_valide_par_essai).fillna(df['Lq_reel'])


df_ut = df_ut.dropna()  # Supprimer les lignes avec des valeurs manquantes
X1 = df_ut.drop(columns=['E1'])
y1 = df_ut['E1']

# 3. Nettoyage final
df_ut = df_ut.dropna()
if df_ut.shape[0] == 0:
    raise ValueError("⚠️ ARRET : Le tableau est vide après le dropna().")
"""
groupes = X1['essai']
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(X1, y1, groups=groupes))

# On génère nos vrais X_train et X_test
X_train, X_test = X1.iloc[train_idx].copy(), X1.iloc[test_idx].copy()
y_train, y_test = y1.iloc[train_idx].copy(), y1.iloc[test_idx].copy()
groupes_train = groupes.iloc[train_idx] 


# 3. CRUCIAL : On supprime la colonne 'essai' pour que le modèle ne triche pas
X_train = X_train.drop(columns=['essai'])
X_test = X_test.drop(columns=['essai'])
"""
"""
X_train, X_test, y_train, y_test = train_test_split(X1.groupby('essai').first(), y1, test_size=0.2, random_state=42)
X_train, X_test = X_train.drop(columns=['essai']), X_test.drop(columns=['essai'])
print(X_train, X_test)
"""

/var/folders/db/0jvml7bx2wj5gv9322q7lczc0000gn/T/ipykernel_69645/3255010232.py:9: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_ut['P1_reel'] = df['essai'].map(p1_v0_par_essai).fillna(df['P1_reel'])
/var/folders/db/0jvml7bx2wj5gv9322q7lczc0000gn/T/ipykernel_69645/3255010232.py:12: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_ut['Lq_reel'] = df['essai'].map(lq_valide_par_essai).fillna(df['Lq_reel'])


"\nX_train, X_test, y_train, y_test = train_test_split(X1.groupby('essai').first(), y1, test_size=0.2, random_state=42)\nX_train, X_test = X_train.drop(columns=['essai']), X_test.drop(columns=['essai'])\nprint(X_train, X_test)\n"

In [14]:
#Recherche des meilleurs hyperparamètres pour le Random Forest sur E1

param_grid = {
    'n_estimators': [100, 200, 600, 800],      # On cherche le bon nombre d'arbres
    'max_depth': [5, 8, 12, None],             # On cherche la bonne profondeur (PAS DE 'None' !)
    'min_samples_leaf': [2, 3, 5]        # On cherche à lisser plus ou moins les feuilles
}

# Modèle de base pour la recherche
rf_base = RandomForestRegressor(random_state=42)

# cv=5 : Validation croisée en 5 blocs. n_jobs=-1 : Utilise tout le CPU.
grid_moy = GridSearchCV(estimator=rf_base, param_grid=param_grid, cv=5, scoring='neg_mean_absolute_error', n_jobs=-1)
grid_moy.fit(X_train, y_train)
rf_moy = grid_moy.best_estimator_
print(f"Meilleurs paramètres E1 : {grid_moy.best_params_}")

print("--- 4. Évaluation ---")
# Prédictions (avec les meilleurs modèles trouvés automatiquement)
pred_moy = rf_moy.predict(X_test)

# Métriques pour E1
print(">> PERFORMANCES SUR E1 (La valeur moyenne) :")
print(f"MAE  : {mean_absolute_error(y_test, pred_moy):.4f}")
print(f"RMSE : {np.sqrt(mean_squared_error(y_test, pred_moy)):.4f}")
print(f"R2   : {r2_score(y_test, pred_moy):.4f}\n")


Recherche des meilleurs paramètres pour E1 en cours...
Meilleurs paramètres E1 : {'max_depth': None, 'min_samples_leaf': 2, 'n_estimators': 800}
--- 4. Évaluation ---
>> PERFORMANCES SUR E1 (La valeur moyenne) :
MAE  : 0.0342
RMSE : 0.0502
R2   : 0.9120



In [ ]:
# Modèle Random Forest
rf = RandomForestRegressor(
    n_estimators=200,   # nombre d'arbres
    max_depth= None,    # profondeur illimitée
    random_state=42,
    n_jobs=-1          # utilise tous les cœurs CPU
)

# Entraînement
rf.fit(X_train, y_train)

# Prédictions
y_pred = rf.predict(X_test)
y_pred = pd.Series(y_pred, index=y_test.index)

# Ajout dans results
#results["RF"] = y_pred

# Métriques
r2 = r2_score(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)

print(f"R2: {r2:.06f}")
print(f"MSE: {mse:.06f}")
print()

mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse) # La racine carrée pour retrouver l'unité physique de E1

print(f"Erreur Absolue Moyenne (MAE) : {mae:.2f}")
print(f"Racine de l'Erreur Quadratique Moyenne (RMSE) : {rmse:.2f} ")
print("\n")

# Exemple de sauvegarde dans un dataframe de résultats
results = pd.DataFrame({'Vraie_Valeur': y_test, 'Prediction_RF': y_pred})
print("Aperçu des prédictions :")
print(results.head())
print("\n")

print("--- 4. Importance des Paramètres ---")
# On récupère le classement d'importance des variables calculé par le modèle
importances = rf.feature_importances_
for col, imp in sorted(zip(X1.columns, importances), key=lambda x: x[1], reverse=True):
    print(f"{col:<10}: {imp*100:.1f}%")





R2: 0.911652
MSE: 0.002525

Erreur Absolue Moyenne (MAE) : 0.03
Racine de l'Erreur Quadratique Moyenne (RMSE) : 0.05 


Aperçu des prédictions :
      Vraie_Valeur  Prediction_RF
1822      0.491788       0.503402
1273      0.619427       0.591598
1321      0.659784       0.666267
599       0.617081       0.630364
1805      0.496481       0.402137


--- 4. Importance des Paramètres ---
V         : 43.5%
P2        : 14.9%
P1_reel   : 12.6%
E2        : 9.0%
P5        : 8.8%
Lq_reel   : 5.7%
P6_reel   : 2.8%
P4        : 2.7%
Lt        : 0.0%


In [ ]:
# === Courbes d'apprentissage : R2 et RMSE en fonction de la taille du train set ===
# On respecte le découpage par groupe (GroupShuffleSplit sur 'essai') pour éviter
# toute fuite de données entre les essais, comme dans le reste du notebook.
import matplotlib.pyplot as plt
from sklearn.model_selection import learning_curve, GroupKFold
from sklearn.base import clone

# Modèle identique à celui entraîné plus haut (non entraîné : learning_curve refait ses fits)
rf_lc = clone(rf)

# CV groupée par essai (nombre de splits limité par le nb d'essais dans le train)
n_groupes = groupes_train.nunique()
cv = GroupKFold(n_splits=min(5, n_groupes))

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
train_sizes = np.linspace(0.1, 1.0, 8)

for ax, scoring, label, is_rmse in [
    (axes[0], "r2", "R2", False),
    (axes[1], "neg_root_mean_squared_error", "RMSE", True),
]:
    sizes, train_scores, val_scores = learning_curve(
        rf_lc, X_train, y_train,
        groups=groupes_train,          # respecte les groupes d'essais
        cv=cv, scoring=scoring,
        train_sizes=train_sizes, n_jobs=-1,
    )
    if is_rmse:                        # neg_root_mean_squared_error -> valeurs négatives
        train_scores, val_scores = -train_scores, -val_scores

    tr_m, tr_s = train_scores.mean(1), train_scores.std(1)
    va_m, va_s = val_scores.mean(1), val_scores.std(1)

    ax.plot(sizes, tr_m, "o-", color="tab:blue", label="Train")
    ax.fill_between(sizes, tr_m - tr_s, tr_m + tr_s, alpha=0.15, color="tab:blue")
    ax.plot(sizes, va_m, "o-", color="tab:orange", label="Validation (CV groupée)")
    ax.fill_between(sizes, va_m - va_s, va_m + va_s, alpha=0.15, color="tab:orange")

    ax.set_xlabel("Taille du jeu d'entraînement")
    ax.set_ylabel(label)
    ax.set_title(f"Convergence {label} - RandomForest (E1)")
    ax.legend(loc="best")
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Lecture :
# - courbes qui convergent haut (R2) / bas (RMSE) = bon ajustement
# - grand écart persistant Train vs Validation = surapprentissage
# - les deux plates et mauvaises = sous-apprentissage / features insuffisantes
